# Create Work Pull Notice Links (oxjob #2264)

One row per candidate pull-notice link: `origin_work_id` (the notice), `target_work_id` (the work it pulls), its link
type (`retraction`, `withdrawal` or `removal`), where it came from, and `served` (it passed every rule and is the one row
for its edge). `notebooks/end2end/CreateWorkLinks` unions the served rows into `openalex.works.work_links` under their
link type; `/links` serves them, and CreateWorksEnriched lists each work's notices in `pull_notice_works`.

Jason approved the shape on 22 Nov 2026 (oxjob #2264: a "pull notice" is a notice that retracts, withdraws or removes a
work; the three link types, their definitions and their crosswalk to 15 other vocabularies are on
help.openalex.org/data/link-types). Sources:
- **Crossref** `update-to` on the notice's record, types `retraction`, `withdrawal`, `removal` (publisher deposits, and
  Retraction Watch's data inside Crossref). A record whose `update-to` names itself is an in-place pull: it sets
  `is_pulled` (CreateWorksBase) but makes no link, since a link joins two works.
- **PubMed** "retraction of" pairs (`openalex.works.notice_link_pairs`, rebuilt by Map_Work_Ids), as `retraction`.
- **Retraction Watch** notice DOI -> paper DOI, nature Retraction, as `retraction`. Retraction Watch and PubMed don't
  tell the three kinds apart; `retraction` is their word for all three, so it is ours when they are the only source.

Rules (the first that fails names the drop): both ends resolve (DOI or PMID locations, then merges) to live works;
not the same work; not two versions of one Cochrane review (a version that withdraws its predecessor is a version, not
a pull); the notice is a notice (typed `retraction` or `erratum`, or titled as one: publishers type many retraction
notices `article` or `erratum`); the notice isn't dated more than a year before the work; a PubMed or Retraction Watch
pair that Crossref types as something else (a correction, an expression of concern) is dropped, unless the notice's
title names a pull and the other type came from the publisher (publishers deposit many retraction notices as
`erratum`). One row per edge is served, by source: the publisher's Crossref deposit, then Retraction Watch in Crossref,
then PubMed, then Retraction Watch. When two records each claim to be the other's notice, the later one is the notice.
Step 5 refuses a build serving 20% fewer links than the last.

Checked before shipping (21-22 Nov 2026): 61,798 works, 62,202 links (retraction 61,672, withdrawal 519, removal 11);
a random 47 served links plus all 11 removals read by hand, all right.

Runs in the Work Links job, before Build_Work_Links. Revert: drop the pull-notice leg in CreateWorkLinks and this task
from `jobs/work_links.yaml`; the tables can be dropped.

In [0]:
-- 1. Every declared notice -> work pair, ids normalized. Crossref pairs of every update type are kept: step 2 uses the
--    other types to judge PubMed and Retraction Watch pairs.
CREATE OR REPLACE TABLE openalex.works.work_pull_notice_links_declared AS
WITH cr AS (
  SELECT lower(native_id) AS notice_id,
         regexp_replace(lower(trim(u.DOI)), '^(https?://(dx[.])?doi[.]org/|doi:)', '') AS paper_id,
         lower(u.type) AS update_type,
         lower(coalesce(u.source, 'publisher')) AS update_source
  FROM openalex.crossref.crossref_deduplicated LATERAL VIEW explode(`update-to`) t AS u
  WHERE size(`update-to`) > 0 AND u.DOI IS NOT NULL
),
rw AS (
  SELECT regexp_replace(lower(trim(retraction_doi)), '^(https?://(dx[.])?doi[.]org/|doi:)', '') AS notice_id,
         regexp_replace(lower(trim(original_paper_doi)), '^(https?://(dx[.])?doi[.]org/|doi:)', '') AS paper_id
  FROM openalex.works.retraction_watch
  WHERE retraction_nature = 'Retraction'
)
SELECT 'crossref' AS declared_by, 'doi' AS ns, notice_id, paper_id, update_type, update_source
FROM cr WHERE notice_id <> paper_id
UNION ALL
SELECT 'pubmed', 'pmid', notice_id, target_id, 'retraction', 'pubmed'
FROM openalex.works.notice_link_pairs
WHERE notice_type = 'retractionof' AND notice_ns = 'pmid' AND target_ns = 'pmid'
UNION ALL
SELECT 'retraction-watch', 'doi', notice_id, paper_id, 'retraction', 'retraction-watch'
FROM rw WHERE notice_id LIKE '10.%' AND paper_id LIKE '10.%' AND notice_id <> paper_id

In [0]:
-- 2. Resolve both ends to live works (DOI or PMID locations, then merges) and apply the rules; the first failing rule
--    names the drop.
CREATE OR REPLACE TABLE openalex.works.work_pull_notice_links_candidates AS
WITH d AS (SELECT * FROM openalex.works.work_pull_notice_links_declared),
ids AS (SELECT ns, notice_id AS id FROM d UNION SELECT ns, paper_id FROM d),
id_works AS (
  SELECT l.native_id_namespace AS ns, l.native_id AS id,
         MAX_BY(l.work_id, CASE l.provenance WHEN 'crossref' THEN 3 WHEN 'datacite' THEN 2 ELSE 1 END) AS work_id
  FROM openalex.works.location_work_ids l
  JOIN ids ON l.native_id_namespace = ids.ns AND l.native_id = ids.id
  WHERE l.work_id IS NOT NULL
    AND (l.native_id_namespace = 'pmid' OR l.provenance IN (/*DOI_RA*/ 'crossref', 'datacite'))
  GROUP BY 1, 2
),
resolved AS (
  SELECT i.ns, i.id, coalesce(m.merge_into_id, i.work_id) AS work_id
  FROM id_works i LEFT JOIN openalex.works.merged_work_ids_map m ON m.id = i.work_id
),
w AS (
  SELECT id, type, lower(title) AS title, publication_date
  FROM openalex.works.openalex_works WHERE id IN (SELECT work_id FROM resolved)
),
j AS (
  SELECT d.*, rn.work_id AS origin_work_id, rp.work_id AS target_work_id,
         wn.id AS notice_live, wp.id AS paper_live, wn.type AS notice_type, wn.title AS notice_title,
         wn.publication_date AS notice_date, wp.publication_date AS paper_date
  FROM d
  LEFT JOIN resolved rn ON rn.ns = d.ns AND rn.id = d.notice_id
  LEFT JOIN resolved rp ON rp.ns = d.ns AND rp.id = d.paper_id
  LEFT JOIN w wn ON wn.id = rn.work_id
  LEFT JOIN w wp ON wp.id = rp.work_id
),
-- Pairs Crossref types as something other than a pull, and whether Retraction Watch (inside Crossref) said so.
cr_other AS (
  SELECT origin_work_id, target_work_id, max(update_source = 'retraction-watch') AS by_rw
  FROM j
  WHERE declared_by = 'crossref' AND update_type NOT IN ('retraction', 'withdrawal', 'removal')
    AND origin_work_id IS NOT NULL AND target_work_id IS NOT NULL
  GROUP BY 1, 2
)
SELECT j.declared_by, j.update_source, j.update_type AS link_type,
       j.origin_work_id, j.target_work_id,
       concat(j.ns, ':', j.notice_id) AS notice_native_id, concat(j.ns, ':', j.paper_id) AS paper_native_id,
       j.notice_date, j.paper_date,
       CASE
         WHEN j.origin_work_id IS NULL THEN 'notice id not in OpenAlex'
         WHEN j.target_work_id IS NULL THEN 'work id not in OpenAlex'
         WHEN j.notice_live IS NULL THEN 'notice not live'
         WHEN j.paper_live IS NULL THEN 'work not live'
         WHEN j.origin_work_id = j.target_work_id THEN 'same work'
         WHEN j.ns = 'doi' AND regexp_extract(j.notice_id, '^10[.]1002/14651858[.]((cd|mr)[0-9]+)', 1) <> ''
              AND regexp_extract(j.notice_id, '^10[.]1002/14651858[.]((cd|mr)[0-9]+)', 1)
                  = regexp_extract(j.paper_id, '^10[.]1002/14651858[.]((cd|mr)[0-9]+)', 1)
           THEN 'versions of one Cochrane review'
         WHEN coalesce(j.notice_type, '') NOT IN ('retraction', 'erratum')
              AND NOT coalesce(j.notice_title, '') RLIKE '^[^a-z0-9]*(retract|withdraw|remov|notice)'
              AND NOT coalesce(j.notice_title, '') RLIKE '(retraction|retracted|notice of|redundant publication|duplicate publication)'
           THEN 'notice is not a notice'
         WHEN year(j.notice_date) < year(j.paper_date) - 1 THEN 'notice dated before the work'
         WHEN j.declared_by <> 'crossref' AND co.origin_work_id IS NOT NULL
              AND (co.by_rw OR NOT coalesce(j.notice_title, '') RLIKE '(retract|withdraw|remov)')
           THEN 'Crossref gives the pair another type'
       END AS drop_reason
FROM j
LEFT JOIN cr_other co ON co.origin_work_id = j.origin_work_id AND co.target_work_id = j.target_work_id
WHERE j.update_type IN ('retraction', 'withdrawal', 'removal')

In [0]:
-- 3. One row per candidate; `served` marks exactly one row per (notice, work) edge that passed every rule, taken by
--    source (the publisher's Crossref deposit, Retraction Watch in Crossref, PubMed, Retraction Watch), which also sets
--    its link type. When two records each claim to be the other's notice, the later-dated one is the notice; a tie
--    serves neither. Built beside the served table, checked, then swapped in.
CREATE OR REPLACE TABLE openalex.works.work_pull_notice_links_build AS
WITH r AS (
  SELECT *, row_number() OVER (
    PARTITION BY origin_work_id, target_work_id
    ORDER BY drop_reason IS NULL DESC,
             CASE WHEN declared_by = 'crossref' AND update_source = 'publisher' THEN 1
                  WHEN declared_by = 'crossref' THEN 2 WHEN declared_by = 'pubmed' THEN 3 ELSE 4 END,
             notice_native_id, paper_native_id) AS rn
  FROM openalex.works.work_pull_notice_links_candidates
),
passed AS (SELECT origin_work_id, target_work_id, notice_date FROM r WHERE rn = 1 AND drop_reason IS NULL),
two_way AS (
  SELECT a.origin_work_id, a.target_work_id
  FROM passed a JOIN passed b ON a.origin_work_id = b.target_work_id AND a.target_work_id = b.origin_work_id
  WHERE NOT (coalesce(a.notice_date, DATE '1000-01-01') > coalesce(b.notice_date, DATE '1000-01-01'))
)
SELECT r.origin_work_id, r.target_work_id, r.link_type, r.declared_by AS link_source, r.update_source,
       (r.drop_reason IS NULL AND r.rn = 1 AND tw.origin_work_id IS NULL
        AND r.origin_work_id IS NOT NULL AND r.target_work_id IS NOT NULL) AS served,
       CASE WHEN r.drop_reason IS NOT NULL THEN r.drop_reason
            WHEN r.rn > 1 THEN 'duplicate edge'
            WHEN tw.origin_work_id IS NOT NULL THEN 'two-way pair: the other record is the notice' END AS drop_reason,
       r.notice_native_id, r.paper_native_id, current_timestamp() AS built_at
FROM r
LEFT JOIN two_way tw ON tw.origin_work_id = r.origin_work_id AND tw.target_work_id = r.target_work_id

In [0]:
-- 4. Fuse: refuse a build that serves 20% fewer links than the one being replaced (a broken input, not real change).
CREATE TABLE IF NOT EXISTS openalex.works.work_pull_notice_links LIKE openalex.works.work_pull_notice_links_build

In [0]:
SELECT assert_true(
  (SELECT count_if(served) FROM openalex.works.work_pull_notice_links_build)
    >= 0.8 * (SELECT count_if(served) FROM openalex.works.work_pull_notice_links),
  'work_pull_notice_links_build serves over 20% fewer pull-notice links than work_pull_notice_links: check the inputs before swapping'
) AS fuse_ok

In [0]:
-- 5. Swap in.
CREATE OR REPLACE TABLE openalex.works.work_pull_notice_links
COMMENT 'oxjob #2264: pull-notice links (origin = the notice, target = the work it retracts, withdraws or removes); served rows go to work_links under their link type'
AS SELECT * FROM openalex.works.work_pull_notice_links_build